# Wikipedia Web Scrapping

## Anggota Kelompok
- Najla Zahira (25031554041)
- Moza Athaillah Azzahra (25031554185)

## HTTP

- Domain topik : AI, Computer Science, Data & Statistic, Technology & Internet
- Objek: Judul, URL, Overview_Isi, Tanggal_Suntingan, dan Penerbit

In [ ]:
pip install beautifulsoup4

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

In [ ]:
category_urls = [
    'https://id.wikipedia.org/wiki/Kategori:Komputasi',
    'https://id.wikipedia.org/wiki/Kategori:Kecerdasan_buatan',
    'https://id.wikipedia.org/wiki/Kategori:Ilmu_komputer',

    'https://id.wikipedia.org/wiki/Kategori:Pengolahan_data',
    'https://id.wikipedia.org/wiki/Kategori:Penyimpanan_data_komputer',
    'https://id.wikipedia.org/wiki/Kategori:Manajemen_data',

    'https://id.wikipedia.org/wiki/Kategori:Teknologi_informasi',
    'https://id.wikipedia.org/wiki/Kategori:Teknologi_digital',
    'https://id.wikipedia.org/wiki/Kategori:Internet'
]

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36'
}

semua_sukses = True
articles_list = []
seen_urls = set()

for idx, url in enumerate(category_urls, start=1):
    try:
        page = requests.get(url, headers=headers, timeout=10)

        if page.status_code == 200:
            print(f"[{idx}/{len(category_urls)}] [200]  -> {url}")

            soup = BeautifulSoup(page.text, 'html.parser')
            container = soup.find('div', id='mw-pages')
            if container:
                for link in container.find_all('a'):
                    href = link.get('href')
                    title = link.text.strip()
                    if href and href.startswith('/wiki/') and not any(p in href for p in [':', 'Istimewa', 'Halaman_Utama']):
                        full_url = 'https://id.wikipedia.org' + href
                        if full_url not in seen_urls:
                            seen_urls.add(full_url)
                            articles_list.append({'Judul': title, 'URL': full_url})
        else:
            print(f"[{idx}/{len(category_urls)}] [{page.status_code} GAGAL] -> {url}")
            semua_sukses = False

    except Exception as e:
        print(f"[{idx}/{len(category_urls)}] [ERROR] -> {url} (Detail: {e})")
        semua_sukses = False

if semua_sukses:
    print("\nSemua URL berhasil")
else:
    print("\nAda URL yang gagal.")

df_urls = pd.DataFrame(articles_list)
print(f"Total URL yang berhasil diekstrak: {len(df_urls)}")

[1/9] [200]  -> https://id.wikipedia.org/wiki/Kategori:Komputasi
[2/9] [200]  -> https://id.wikipedia.org/wiki/Kategori:Kecerdasan_buatan
[3/9] [200]  -> https://id.wikipedia.org/wiki/Kategori:Ilmu_komputer
[4/9] [200]  -> https://id.wikipedia.org/wiki/Kategori:Pengolahan_data
[5/9] [200]  -> https://id.wikipedia.org/wiki/Kategori:Penyimpanan_data_komputer
[6/9] [200]  -> https://id.wikipedia.org/wiki/Kategori:Manajemen_data
[7/9] [200]  -> https://id.wikipedia.org/wiki/Kategori:Teknologi_informasi
[8/9] [200]  -> https://id.wikipedia.org/wiki/Kategori:Teknologi_digital
[9/9] [200]  -> https://id.wikipedia.org/wiki/Kategori:Internet

Semua URL berhasil
Total URL yang berhasil diekstrak: 479


In [ ]:
scraped_data_10 = []

for idx, item in enumerate(articles_list[:10], start=1):
    url = item['URL']
    try:
        res = requests.get(url, headers=headers, timeout=10)
        if res.status_code == 200:
            soup = BeautifulSoup(res.text, 'html.parser')

            h1_title = soup.find('h1', id='firstHeading')
            title_text = h1_title.text.strip() if h1_title else item['Judul']

            last_mod = soup.find('li', id='footer-info-lastmod')
            if last_mod:
                raw_text = last_mod.text.strip()
                if 'pada ' in raw_text:
                    date_part = raw_text.split('pada ')[1]
                    clean_date = date_part.split(',')[0].strip()
                else:
                    clean_date = raw_text
            else:
                clean_date = "Tidak ditemukan"

            content_div = soup.find('div', class_='mw-parser-output')
            paragraphs = []
            if content_div:
                for p in content_div.find_all('p'):
                    text = p.text.strip()
                    if text:
                        paragraphs.append(text)

            full_overview = " ".join(paragraphs)

            scraped_data_10.append({
                'Judul': title_text,
                'URL': url,
                'Overview_Isi': full_overview,
                'Tanggal_Suntingan': clean_date,
                'Penerbit': 'Wikipedia Bahasa Indonesia'
            })
    except Exception:
        pass
    time.sleep(0.2)

print("HASIL SCRAPING 10 BARIS TERATAS:\n")
df_sample10 = pd.DataFrame(scraped_data_10)
df_sample10

HASIL SCRAPING 10 BARIS TERATAS:



,Judul,URL,Overview_Isi,Tanggal_Suntingan,Penerbit
0,Algoritma bias,https://id.wikipedia.org/wiki/Algoritma_bias,Algoritma bias menggambarkan galat sistematis ...,1 Juni 2026,Wikipedia Bahasa Indonesia
1,Algoritma Lanczos,https://id.wikipedia.org/wiki/Algoritma_Lanczos,Algoritma Lanczos adalah algoritma iteratif ad...,25 September 2026,Wikipedia Bahasa Indonesia
2,ASCII,https://id.wikipedia.org/wiki/ASCII,(ASCII) singkatan dari American Standard Code ...,17 September 2026,Wikipedia Bahasa Indonesia
3,Cermin (komputasi),https://id.wikipedia.org/wiki/Cermin_(komputasi),"Dalam komputasi, sebuah cermin (bahasa Inggris...",27 Mei 2024,Wikipedia Bahasa Indonesia
4,Karakter null,https://id.wikipedia.org/wiki/Karakter_null,"Karakter null (atau bita null), disingkat NUL ...",17 September 2026,Wikipedia Bahasa Indonesia
5,Karakter ujung teks,https://id.wikipedia.org/wiki/Karakter_ujung_teks,Karakter ujung teks (ETX) (nilai heksadesimaln...,26 Juni 2026,Wikipedia Bahasa Indonesia
6,ASCII,https://id.wikipedia.org/wiki/Kode_ASCII,(ASCII) singkatan dari American Standard Code ...,17 September 2026,Wikipedia Bahasa Indonesia
7,Kode duplikat,https://id.wikipedia.org/wiki/Kode_duplikat,Kode duplikat merupakan sumber kode yang diran...,5 April 2024,Wikipedia Bahasa Indonesia
8,Komputasi optis,https://id.wikipedia.org/wiki/Komputasi_optis,Komputasi optis adalah penggunaan foton yang d...,19 Oktober 2025,Wikipedia Bahasa Indonesia
9,LIPI Public Cluster,https://id.wikipedia.org/wiki/LIPI_Public_Cluster,LIPI Public Cluster atau LPC adalah sarana kom...,10 Februari 2026,Wikipedia Bahasa Indonesia


In [ ]:
scraped_data_all = []

for idx, item in enumerate(articles_list, start=1):
    url = item['URL']
    try:
        res = requests.get(url, headers=headers, timeout=10)
        if res.status_code == 200:
            soup = BeautifulSoup(res.text, 'html.parser')

            h1_title = soup.find('h1', id='firstHeading')
            title_text = h1_title.text.strip() if h1_title else item['Judul']

            last_mod = soup.find('li', id='footer-info-lastmod')
            if last_mod:
                raw_text = last_mod.text.strip()
                if 'pada ' in raw_text:
                    date_part = raw_text.split('pada ')[1]
                    clean_date = date_part.split(',')[0].strip()
                else:
                    clean_date = raw_text
            else:
                clean_date = "Tidak ditemukan"

            content_div = soup.find('div', class_='mw-parser-output')
            paragraphs = []
            if content_div:
                for p in content_div.find_all('p'):
                    text = p.text.strip()
                    if text:
                        paragraphs.append(text)

            full_overview = " ".join(paragraphs)

            scraped_data_all.append({
                'Judul': title_text,
                'URL': url,
                'Overview_Isi': full_overview,
                'Tanggal_Suntingan': clean_date,
                'Penerbit': 'Wikipedia Bahasa Indonesia'
            })
            print(f"[{idx}/{len(articles_list)}] Berhasil: {title_text}")

    except Exception as e:
        print(f"[{idx}/{len(articles_list)}] Gagal {url}: {e}")

    time.sleep(0.3)

df_result = pd.DataFrame(scraped_data_all)

print("\nScraping Selesai")

[1/479] Berhasil: Algoritma bias
[2/479] Berhasil: Algoritma Lanczos
[3/479] Berhasil: ASCII
[4/479] Berhasil: Cermin (komputasi)
[5/479] Berhasil: Karakter null
[6/479] Berhasil: Karakter ujung teks
[7/479] Berhasil: ASCII
[8/479] Berhasil: Kode duplikat
[9/479] Berhasil: Komputasi optis
[10/479] Berhasil: LIPI Public Cluster
[11/479] Berhasil: Membangkitkan Kombinasi
[12/479] Berhasil: Komputasi neuromorfik
[13/479] Berhasil: Teori otomata
[14/479] Berhasil: Uptime
[15/479] Berhasil: Kecerdasan buatan
[16/479] Berhasil: 2005 DARPA Grand Challenge
[17/479] Berhasil: Agen fuzzy
[18/479] Berhasil: Agen otonom
[19/479] Berhasil: Agen perangkat lunak
[20/479] Berhasil: AI Dungeon
[21/479] Berhasil: AI Love You (film 2022)
[22/479] Berhasil: AI winter
[23/479] Berhasil: AI-komplit
[24/479] Berhasil: Akinator
[25/479] Berhasil: Alat Ebert
[26/479] Berhasil: Algoritma genetik
[27/479] Berhasil: Algoritma semut
[28/479] Berhasil: Alice (asisten virtual)
[29/479] Berhasil: AlphaGo
[30/479] Ber

## Quality Filtering

In [ ]:
df_result

,Judul,URL,Overview_Isi,Tanggal_Suntingan,Penerbit
0,Algoritma bias,https://id.wikipedia.org/wiki/Algoritma_bias,Algoritma bias menggambarkan galat sistematis ...,1 Juni 2026,Wikipedia Bahasa Indonesia
1,Algoritma Lanczos,https://id.wikipedia.org/wiki/Algoritma_Lanczos,Algoritma Lanczos adalah algoritma iteratif ad...,25 September 2026,Wikipedia Bahasa Indonesia
2,ASCII,https://id.wikipedia.org/wiki/ASCII,(ASCII) singkatan dari American Standard Code ...,17 September 2026,Wikipedia Bahasa Indonesia
3,Cermin (komputasi),https://id.wikipedia.org/wiki/Cermin_(komputasi),"Dalam komputasi, sebuah cermin (bahasa Inggris...",27 Mei 2024,Wikipedia Bahasa Indonesia
4,Karakter null,https://id.wikipedia.org/wiki/Karakter_null,"Karakter null (atau bita null), disingkat NUL ...",17 September 2026,Wikipedia Bahasa Indonesia
...,...,...,...,...,...
474,Web Hypertext Application Technology Working G...,https://id.wikipedia.org/wiki/Web_Hypertext_Ap...,Web Hypertext Application Technology Working G...,16 Desember 2022,Wikipedia Bahasa Indonesia
475,Layanan hos web,https://id.wikipedia.org/wiki/Webhosting,Layanan hos web (bahasa Inggris: Web hosting s...,5 September 2025,Wikipedia Bahasa Indonesia
476,Webtun,https://id.wikipedia.org/wiki/Webtun,Webtun (bahasa Korea: 웹툰; bahasa Inggris: webt...,18 September 2026,Wikipedia Bahasa Indonesia
477,Wisaya (perangkat lunak),https://id.wikipedia.org/wiki/Wisaya_(perangka...,Wisaya atau pengatur pasang (bahasa Inggris: w...,18 September 2026,Wikipedia Bahasa Indonesia


In [ ]:
df_result.shape

(479, 5)

In [ ]:
df_result.head()

,Judul,URL,Overview_Isi,Tanggal_Suntingan,Penerbit
0,Algoritma bias,https://id.wikipedia.org/wiki/Algoritma_bias,Algoritma bias menggambarkan galat sistematis ...,1 Juni 2026,Wikipedia Bahasa Indonesia
1,Algoritma Lanczos,https://id.wikipedia.org/wiki/Algoritma_Lanczos,Algoritma Lanczos adalah algoritma iteratif ad...,25 September 2026,Wikipedia Bahasa Indonesia
2,ASCII,https://id.wikipedia.org/wiki/ASCII,(ASCII) singkatan dari American Standard Code ...,17 September 2026,Wikipedia Bahasa Indonesia
3,Cermin (komputasi),https://id.wikipedia.org/wiki/Cermin_(komputasi),"Dalam komputasi, sebuah cermin (bahasa Inggris...",27 Mei 2024,Wikipedia Bahasa Indonesia
4,Karakter null,https://id.wikipedia.org/wiki/Karakter_null,"Karakter null (atau bita null), disingkat NUL ...",17 September 2026,Wikipedia Bahasa Indonesia


Lihat duplikasi

In [ ]:
print("jumlah data:", len(df_result))
print("Judul unik:", df_result['Judul'].nunique())
print("URL unik:", df_result['URL'].nunique())

jumlah data: 479
Judul unik: 475
URL unik: 479


Melihat judul yang muncul lebih dari sekali

In [ ]:
duplicate_titles = df_result[
    df_result['Judul'].duplicated(keep=False)
].sort_values('Judul')

duplicate_titles[['Judul', 'URL']]

,Judul,URL
2,ASCII,https://id.wikipedia.org/wiki/ASCII
6,ASCII,https://id.wikipedia.org/wiki/Kode_ASCII
397,Layanan hos web,https://id.wikipedia.org/wiki/Layanan_hos_web
475,Layanan hos web,https://id.wikipedia.org/wiki/Webhosting
402,Milis,https://id.wikipedia.org/wiki/Mailing_list
407,Milis,https://id.wikipedia.org/wiki/Milis
311,Teknologi informasi komunikasi,https://id.wikipedia.org/wiki/Teknologi_Inform...
312,Teknologi informasi komunikasi,https://id.wikipedia.org/wiki/Teknologi_inform...


Melihat distribusi panjang artikel

In [ ]:
df_result['Overview_Isi'].str.len().describe()

,Overview_Isi
count,479.000000
mean,4030.423800
std,4229.138767
min,0.000000
25%,1134.000000
50%,2392.000000
75%,5482.500000
max,24241.000000


Distribusi threshold

In [ ]:
thresholds = [0, 500, 1000, 1500, 2000, 2500, 3000]

for t in thresholds:
    jumlah = (df_result['Overview_Isi'].str.len() >= t).sum()
    print(f"Minimal {t:>4} karakter → {jumlah} artikel")

Minimal    0 karakter → 479 artikel
Minimal  500 karakter → 453 artikel
Minimal 1000 karakter → 374 artikel
Minimal 1500 karakter → 311 artikel
Minimal 2000 karakter → 271 artikel
Minimal 2500 karakter → 232 artikel
Minimal 3000 karakter → 207 artikel


Artikel dengan teks <500 karakter

In [ ]:
df_result['Panjang_Teks'] = df_result['Overview_Isi'].str.len()

df_result[
    df_result['Panjang_Teks'] < 500
][['Judul', 'URL', 'Panjang_Teks']]

,Judul,URL,Panjang_Teks
16,Agen fuzzy,https://id.wikipedia.org/wiki/Agen_fuzzy,378
32,Brainrot Italia,https://id.wikipedia.org/wiki/Brainrot_Italia,0
33,CarynAI,https://id.wikipedia.org/wiki/CarynAI,420
35,Colossus (superkomputer),https://id.wikipedia.org/wiki/Colossus_(superk...,413
45,European High Performance Computing Joint Unde...,https://id.wikipedia.org/wiki/European_High_Pe...,0
60,Inflection AI,https://id.wikipedia.org/wiki/Inflection_AI,410
154,Bahasa Dasar Sistem Informasi,https://id.wikipedia.org/wiki/Bahasa_Dasar_Sis...,231
172,Occam-π,https://id.wikipedia.org/wiki/Occam-%CF%80,413
179,Penguraian (komputasi),https://id.wikipedia.org/wiki/Penguraian_(komp...,382
181,Peta memori,https://id.wikipedia.org/wiki/Peta_memori,426


In [ ]:
articles_list[0]

{'Judul': 'Algoritma bias',
 'URL': 'https://id.wikipedia.org/wiki/Algoritma_bias'}

In [ ]:
articles_list[1]

{'Judul': 'Algoritma Lanczos',
 'URL': 'https://id.wikipedia.org/wiki/Algoritma_Lanczos'}

Kandidat 300 artikel dari hasil threshold >= 1500

In [ ]:
df_candidate = df_result[
    df_result['Panjang_Teks'] >= 1500
].copy()

print("Jumlah kandidat:", len(df_candidate))

Jumlah kandidat: 311


In [ ]:
df_candidate.head()

,Judul,URL,Overview_Isi,Tanggal_Suntingan,Penerbit,Panjang_Teks
0,Algoritma bias,https://id.wikipedia.org/wiki/Algoritma_bias,Algoritma bias menggambarkan galat sistematis ...,1 Juni 2026,Wikipedia Bahasa Indonesia,9968
1,Algoritma Lanczos,https://id.wikipedia.org/wiki/Algoritma_Lanczos,Algoritma Lanczos adalah algoritma iteratif ad...,25 September 2026,Wikipedia Bahasa Indonesia,10448
2,ASCII,https://id.wikipedia.org/wiki/ASCII,(ASCII) singkatan dari American Standard Code ...,17 September 2026,Wikipedia Bahasa Indonesia,2153
4,Karakter null,https://id.wikipedia.org/wiki/Karakter_null,"Karakter null (atau bita null), disingkat NUL ...",17 September 2026,Wikipedia Bahasa Indonesia,1762
6,ASCII,https://id.wikipedia.org/wiki/Kode_ASCII,(ASCII) singkatan dari American Standard Code ...,17 September 2026,Wikipedia Bahasa Indonesia,2153


In [ ]:
df_candidate[['Judul', 'Panjang_Teks']].head(20)

,Judul,Panjang_Teks
0,Algoritma bias,9968
1,Algoritma Lanczos,10448
2,ASCII,2153
4,Karakter null,1762
6,ASCII,2153
8,Komputasi optis,1552
10,Membangkitkan Kombinasi,1516
11,Komputasi neuromorfik,1598
12,Teori otomata,1972
14,Kecerdasan buatan,7925


Klasifikasi berdasarkan kata pada judul

In [ ]:
import re

def klasifikasi_topik(judul):
    judul = judul.lower()

    # 1. Artificial Intelligence
    if re.search(
        r'ai|kecerdasan buatan|artificial intelligence|'
        r'machine learning|pembelajaran mesin|deep learning|'
        r'jaringan saraf|neural network|computer vision|'
        r'natural language processing|pemrosesan bahasa alami|'
        r'robot|robotika|fuzzy',
        judul
    ):
        return 'AI'

    # 2. Data & Statistics
    elif re.search(
        r'data|basis data|database|statistik|statistika|'
        r'regresi|probabilitas|korelasi|data mining|'
        r'data warehouse|data science|analisis data|'
        r'sistem basis data',
        judul
    ):
        return 'Data & Statistics'

    # 3. Computer Science & Computing
    elif re.search(
        r'algoritma|komputasi|komputer|pemrograman|programming|'
        r'kode|coding|perangkat lunak|software|'
        r'sistem operasi|arsitektur komputer|struktur data|'
        r'compiler|kompilator|bahasa pemrograman|'
        r'kriptografi|jaringan komputer|protokol|'
        r'cpu|prosesor|superkomputer',
        judul
    ):
        return 'Computer Science'

    # 4. Technology, Internet & Digital
    elif re.search(
        r'teknologi|internet|web|digital|informasi|'
        r'cloud|awan|telekomunikasi|jaringan|'
        r'website|situs|media sosial|online|'
        r'http|https|email|server|hosting',
        judul
    ):
        return 'Technology & Internet'

    else:
        return 'Other'

In [ ]:
df_candidate['Topik_Sementara'] = df_candidate['Judul'].apply(
    klasifikasi_topik
)

In [ ]:
df_candidate.columns

Index(['Judul', 'URL', 'Overview_Isi', 'Tanggal_Suntingan', 'Penerbit',
       'Panjang_Teks', 'Topik_Sementara'],
      dtype='object')

In [ ]:
df_candidate['Topik_Sementara'].value_counts()

,count
Topik_Sementara,
Other,164
Technology & Internet,52
AI,47
Computer Science,25
Data & Statistics,23


In [ ]:
df_candidate[
    df_candidate['Topik_Sementara'] == 'Other'
][['Judul', 'Panjang_Teks']]

,Judul,Panjang_Teks
2,ASCII,2153
4,Karakter null,1762
6,ASCII,2153
10,Membangkitkan Kombinasi,1516
12,Teori otomata,1972
...,...,...
462,Tombol Akses Terbuka,3268
464,Ujaran kebencian,3501
465,United States v. Microsoft Corp.,3311
469,Vlog,12275


Interprestasi : Ternyata masih ada duplikat dan jumlah pemilihan topik belum seimbang

In [ ]:
df_candidate = df_candidate.drop_duplicates(
    subset=['Judul'],
    keep='first'
).copy()

print("Jumlah setelah menghapus judul duplikat:", len(df_candidate))

Jumlah setelah menghapus judul duplikat: 307


In [ ]:
df_candidate[['Judul', 'URL']].head(20)

,Judul,URL
0,Algoritma bias,https://id.wikipedia.org/wiki/Algoritma_bias
1,Algoritma Lanczos,https://id.wikipedia.org/wiki/Algoritma_Lanczos
2,ASCII,https://id.wikipedia.org/wiki/ASCII
4,Karakter null,https://id.wikipedia.org/wiki/Karakter_null
8,Komputasi optis,https://id.wikipedia.org/wiki/Komputasi_optis
10,Membangkitkan Kombinasi,https://id.wikipedia.org/wiki/Membangkitkan_Ko...
11,Komputasi neuromorfik,https://id.wikipedia.org/wiki/Komputasi_neurom...
12,Teori otomata,https://id.wikipedia.org/wiki/Teori_otomata
14,Kecerdasan buatan,https://id.wikipedia.org/wiki/Kecerdasan_buatan
17,Agen otonom,https://id.wikipedia.org/wiki/Agen_otonom


Buat clasifier baru

In [ ]:
import re

def klasifikasi_topik(judul):
    judul = judul.lower().strip()


    # 1. ARTIFICIAL INTELLIGENCE
    if re.search(
        r'\b(ai|kecerdasan buatan|artificial intelligence|'
        r'machine learning|pembelajaran mesin|deep learning|'
        r'jaringan saraf|neural network|computer vision|'
        r'natural language processing|pemrosesan bahasa alami|'
        r'nlp|robotika|robot|fuzzy|sistem pakar)\b',
        judul
    ):
        return 'AI'

    # 2. DATA & STATISTICS
    elif re.search(
        r'\b(data science|ilmu data|data mining|'
        r'data warehouse|big data|database|basis data|'
        r'pengolahan data|analisis data|'
        r'statistik|statistika|regresi|probabilitas|'
        r'korelasi|distribusi statistik)\b',
        judul
    ):
        return 'Data & Statistics'

    # 3. COMPUTER SCIENCE
    elif re.search(
        r'\b(algoritma|komputasi|komputer|'
        r'pemrograman|programming|program|'
        r'perangkat lunak|software|'
        r'struktur data|otomata|automata|'
        r'teori komputasi|bahasa pemrograman|'
        r'compiler|kompilator|'
        r'sistem operasi|arsitektur komputer|'
        r'prosesor|processor|cpu|gpu|'
        r'memori komputer|kriptografi|'
        r'karakter null|ascii|bit|byte|'
        r'kode komputer|sistem komputer)\b',
        judul
    ):
        return 'Computer Science'

    # 4. TECHNOLOGY & INTERNET
    elif re.search(
        r'\b(teknologi informasi|teknologi digital|'
        r'teknologi|internet|internet|web|website|'
        r'world wide web|www|browser|'
        r'cloud|komputasi awan|server|hosting|'
        r'http|https|email|'
        r'telekomunikasi|aplikasi|'
        r'smartphone|ponsel|media sosial|'
        r'digital|jaringan komputer|'
        r'jaringan internet|wap)\b',
        judul
    ):
        return 'Technology & Internet'

    else:
        return 'Other'

In [ ]:
df_candidate['Topik_Sementara'] = df_candidate['Judul'].apply(
    klasifikasi_topik
)

In [ ]:
df_candidate['Topik_Sementara'].value_counts()

,count
Topik_Sementara,
Other,196
AI,39
Technology & Internet,38
Computer Science,28
Data & Statistics,6


In [ ]:
other_titles = df_candidate[
    df_candidate['Topik_Sementara'] == 'Other'
]['Judul'].tolist()

for i, judul in enumerate(other_titles, 1):
    print(f"{i}. {judul}")

1. Membangkitkan Kombinasi
2. Agen otonom
3. Akinator
4. Alat Ebert
5. Alice (asisten virtual)
6. Apple Intelligence
7. Asisten Google
8. Claude (model bahasa)
9. Cruel World of Dreams and Fears
10. Destination Earth
11. Empati buatan
12. EU-SysFlex
13. Future of Life Institute
14. Google Klip
15. H (perusahaan)
16. Hak atas realitas
17. Imagen (model teks-ke-gambar)
18. Inisiatif Pencitraan Kanker Eropa
19. Institut Penelitian Kecerdasan Mesin
20. Kebijaksanaan buatan
21. Kebodohan buatan
22. Kedekatan buatan
23. Kematian Elaine Herzberg
24. Keruntuhan model
25. Kesadaran buatan
26. Komunitas rasionalis
27. Krisis pasokan memori global 2025-sekarang
28. Kórsafn
29. Masyarakat 5.0
30. Mata v. Avianca, Inc.
31. Midjourney
32. Model teks-ke-gambar
33. Muse Spark
34. OpenAI
35. Otak buatan
36. PauseAI
37. Pemahaman bahasa alami
38. Pengenalan tulisan tangan
39. Pusat Rasionalitas Terapan
40. Recraft
41. Representasi dan Penalaran Pengetahuan
42. Riffusion
43. Roborace
44. Sastra generatif

Melihat seluruh judul artikel dari 296 kandidat artikel (tanpa duplikat)

In [ ]:
for i, judul in enumerate(df_candidate['Judul'].tolist()[:100], 1):
    print(f"{i}. {judul}")

1. Algoritma bias
2. Algoritma Lanczos
3. ASCII
4. Karakter null
5. Komputasi optis
6. Membangkitkan Kombinasi
7. Komputasi neuromorfik
8. Teori otomata
9. Kecerdasan buatan
10. Agen otonom
11. AI Dungeon
12. AI Love You (film 2022)
13. AI winter
14. Akinator
15. Alat Ebert
16. Algoritma genetik
17. Algoritma semut
18. Alice (asisten virtual)
19. Aplikasi kecerdasan buatan untuk militer
20. Apple Intelligence
21. Asisten Google
22. Claude (model bahasa)
23. Cruel World of Dreams and Fears
24. Dampak lingkungan dari kecerdasan buatan
25. Deep Learning (South Park)
26. Destination Earth
27. Diella (sistem AI)
28. Efek kecerdasan buatan
29. Empati buatan
30. Etika dalam kecerdasan buatan
31. EU-SysFlex
32. Figure AI
33. Future of Life Institute
34. Galaxy AI
35. GOLOG (bahasa pemrograman)
36. Google AI Studio
37. Google Klip
38. H (perusahaan)
39. Hak atas realitas
40. Halusinasi (kecerdasan buatan)
41. Imagen (model teks-ke-gambar)
42. Inisiatif Pencitraan Kanker Eropa
43. Institut Penel

In [ ]:
for i, judul in enumerate(df_candidate['Judul'].tolist()[100:200], 101):
    print(f"{i}. {judul}")

101. Terjemahan mesin
102. The Alignment Problem
103. Théâtre D'opéra Spatial
104. Unit pemroses grafis
105. Visi komputer
106. Ilmu komputer
107. ACID
108. Algoritma pencarian
109. Aljabar Boole
110. Analis sistem
111. Analisis leksikal
112. Chieko Asakawa
113. Bahasa pemrograman domain khusus
114. Diagram alir data
115. Hukum Moore
116. Ilmu data
117. Informatika kedokteran
118. Kernel (ilmu komputer)
119. Kode
120. Kontrak pintar
121. NURBS
122. Operasi ternary
123. Penambangan data pendidikan
124. Penyibakan layar
125. Pohon urai
126. Pola rancangan
127. Pola serangan
128. PSI Komp 80
129. Psikoinformatika
130. Randy Pausch
131. Semantika (linguistika)
132. Simulasi komputer
133. Pengolahan data
134. Arsitektur Lambda
135. Modus (statistika)
136. Pengolahan data elektronik
137. Memori volatil
138. Nano-RAM
139. Manajemen data
140. Abstraksi (ilmu komputer)
141. Cambridge Analytica
142. Data tertaut
143. Hak cipta
144. Injeksi SQL
145. Inteligensi bisnis
146. Kolaborasi data
147. Ma

In [ ]:
for i, judul in enumerate(df_candidate['Judul'].tolist()[200:], 201):
    print(f"{i}. {judul}")

201. Pusat Inovasi Digital Eropa
202. Scenario Based Requirement Analysis Method
203. School on Internet Asia
204. Sistem e-Pengadaan Pemerintah
205. Sistem informasi pengelolaan
206. Sistem informasi penumpang
207. Stasiun pemancar-penerima dasar
208. Summly
209. Teknologi informasi komunikasi
210. Teknologi pencari
211. Transformasi digital
212. Tren IT
213. Virtualisasi mejantara
214. WiMAX
215. XDSL
216. Digitisasi
217. Dompet Identitas Digital Eropa
218. FaceTime
219. Identitas digital
220. Keterjangkauan digital
221. Scanlation
222. Sekolah Murid Merdeka
223. Warga digital
224. 1.1.1.1
225. 11B-X-1371
226. 3.5G
227. 4shared
228. Aktivisme Internet
229. Asymmetric digital subscriber line
230. Bandcamp
231. Banjir informasi
232. Biro 1440
233. Bitly
234. Budaya internet
235. Citizendium
236. Deezer
237. Demokrasi maya
238. Digibiz
239. Dunia maya
240. E-PR
241. Go.com
242. Hak digital
243. IndosatM2 Broom
244. Internet untuk Segala
245. Is Google Making Us Stupid?
246. Daftar istil

Buat daftar 25 artikel AI

In [ ]:
ai = [
    "Kecerdasan buatan",
    "Agen otonom",
    "Algoritma genetik",
    "Algoritma semut",
    "Apple Intelligence",
    "Claude (model bahasa)",
    "Dampak lingkungan dari kecerdasan buatan",
    "Etika dalam kecerdasan buatan",
    "Halusinasi (kecerdasan buatan)",
    "Jaringan saraf tiruan",
    "Keamanan kecerdasan buatan",
    "Penerapan kecerdasan buatan",
    "Kecerdasan buatan generatif",
    "Kecerdasan buatan simbolik",
    "Keruntuhan model",
    "Kesadaran buatan",
    "Masalah pengendalian kecerdasan buatan",
    "Midjourney",
    "Model teks-ke-gambar",
    "OpenAI",
    "Pemahaman bahasa alami",
    "Pengenalan tulisan tangan",
    "Representasi dan Penalaran Pengetahuan",
    "Stable Diffusion",
    "Visi komputer"
]

print("Jumlah AI:", len(ai))

Jumlah AI: 25


Buat daftar 30 Computer Science

In [ ]:
computer_science = [
    "Algoritma bias",
    "Algoritma Lanczos",
    "ASCII",
    "Karakter null",
    "Komputasi optis",
    "Membangkitkan Kombinasi",
    "Komputasi neuromorfik",
    "Teori otomata",
    "Ilmu komputer",
    "Algoritma pencarian",
    "Aljabar Boole",
    "Analisis leksikal",
    "Bahasa pemrograman domain khusus",
    "Kernel (ilmu komputer)",
    "Kode",
    "Operasi ternary",
    "Pohon urai",
    "Pola rancangan",
    "Simulasi komputer",
    "Abstraksi (ilmu komputer)",
    "GOLOG (bahasa pemrograman)",
    "Semafor (pemrograman)",
    "General protection fault",
    "BIOS",
    "Hukum Moore",
    "Unit pemroses grafis",
    "Sistem pakar",
    "NURBS",
    "Pola serangan",
    "Akselerasi perangkat keras"
]

print("Jumlah Computer Science:", len(computer_science))

Jumlah Computer Science: 30


Buat daftar 20 Data & Statistic

In [ ]:
data_statistics = [
    "Ilmu data",
    "Diagram alir data",
    "Penambangan data pendidikan",
    "Pengolahan data",
    "Modus (statistika)",
    "Pengolahan data elektronik",
    "Manajemen data",
    "Data tertaut",
    "Injeksi SQL",
    "Inteligensi bisnis",
    "Kolaborasi data",
    "Mahadata",
    "Manajemen pusat data",
    "Normalisasi pangkalan data",
    "Open Database Connectivity",
    "Pangkalan data berorientasi dokumen",
    "Pemrosesan analitik daring",
    "Pemulihan data",
    "Penggalian data",
    "Transaksi basis data"
]

print("Jumlah Data & Statistics:", len(data_statistics))

Jumlah Data & Statistics: 20


Buat daftar 25 Technology & Internet

In [ ]:
technology_internet = [
    "Teknologi informasi",
    "Arsitektur informasi",
    "Deep Web",
    "Enkripsi surel",
    "Etika informasi",
    "Etika siber",
    "Jaringan komputer",
    "Kolaborasi awan",
    "Konversi data",
    "Layanan elektronik",
    "Mode transfer asinkron",
    "Pemasaran Internet",
    "Penggalian web",
    "Perangkat lunak enkripsi",
    "Perdagangan elektronik",
    "Permasalahan pada komputasi awan",
    "Pertukaran data elektronik",
    "Teknologi informasi komunikasi",
    "Transformasi digital",
    "WiMAX",
    "XDSL",
    "Asymmetric digital subscriber line",
    "Internet untuk Segala",
    "Keamanan jaringan",
    "Peramban web"
]

print("Jumlah Technology & Internet:", len(technology_internet))

Jumlah Technology & Internet: 25


In [ ]:
judul_final = ai + computer_science + data_statistics + technology_internet
df_final = df_candidate[
    df_candidate["Judul"].isin(judul_final)
].copy()

In [ ]:
print("Jumlah artikel final:", len(df_final))

Jumlah artikel final: 100


Pastikan tidak ada judul yang hilang

In [ ]:
judul_tidak_ditemukan = set(judul_final) - set(df_final["Judul"])

print("Jumlah judul yang tidak ditemukan:", len(judul_tidak_ditemukan))

for judul in sorted(judul_tidak_ditemukan):
    print("-", judul)

Jumlah judul yang tidak ditemukan: 0


Tambahkan kategori final

In [ ]:
kategori_final = {}

for judul in ai:
    kategori_final[judul] = "AI"

for judul in computer_science:
    kategori_final[judul] = "Computer Science"

for judul in data_statistics:
    kategori_final[judul] = "Data & Statistics"

for judul in technology_internet:
    kategori_final[judul] = "Technology & Internet"

In [ ]:
df_final["Topik"] = df_final["Judul"].map(kategori_final)

print("Jumlah artikel:", len(df_final))
print("Judul unik:", df_final["Judul"].nunique())
print("URL unik:", df_final["URL"].nunique())
print()
print(df_final["Topik"].value_counts())

Jumlah artikel: 100
Judul unik: 100
URL unik: 100

Topik
Computer Science         30
AI                       25
Technology & Internet    25
Data & Statistics        20
Name: count, dtype: int64


### Extracting hasil scraping HTTP to .csv

In [111]:
df_final_export = df_final[['Judul', 'Overview_Isi', 'URL']].copy()

df_final_export['topik'] = df_final_export['Judul'].map(kategori_final)

df_final_export.columns = ['judul', 'teks', 'url', 'topik']

df_final_export.insert(0, 'no', range(1, len(df_final_export) + 1))

df_final_export.to_csv('hasil_scraping_http.csv', index=False)

print("Berhasil menyimpan df_final dengan kolom topik ke 'hasil_scraping_http.csv'!")
print(df_final_export.head())

Berhasil menyimpan df_final dengan kolom topik ke 'hasil_scraping_http.csv'!
   no              judul                                               teks  \
0   1     Algoritma bias  Algoritma bias menggambarkan galat sistematis ...   
1   2  Algoritma Lanczos  Algoritma Lanczos adalah algoritma iteratif ad...   
2   3              ASCII  (ASCII) singkatan dari American Standard Code ...   
4   4      Karakter null  Karakter null (atau bita null), disingkat NUL ...   
8   5    Komputasi optis  Komputasi optis adalah penggunaan foton yang d...   

                                               url             topik  
0     https://id.wikipedia.org/wiki/Algoritma_bias  Computer Science  
1  https://id.wikipedia.org/wiki/Algoritma_Lanczos  Computer Science  
2              https://id.wikipedia.org/wiki/ASCII  Computer Science  
4      https://id.wikipedia.org/wiki/Karakter_null  Computer Science  
8    https://id.wikipedia.org/wiki/Komputasi_optis  Computer Science  


## API


In [ ]:
import requests
import time
import pandas as pd

api_url = "https://id.wikipedia.org/w/api.php"

headers = {
    "User-Agent": "PemrosesanTeks-UNESA/1.0 (student course project)"
}

Fungsi untuk mengambil satu artikel Wikipedia melalui MediaWiki API dengan mekanisme retry dan penanganan rate limit

In [ ]:


def get_wikipedia_api_safe(judul, max_retries=5):

    params = {
        "action": "query",
        "prop": "extracts",
        "explaintext": True,
        "exsectionformat": "plain",
        "titles": judul,
        "format": "json"
    }

    for attempt in range(max_retries):

        response = requests.get(
            api_url,
            params=params,
            headers=headers,
            timeout=60
        )

        # Berhasil
        if response.status_code == 200:

            data = response.json()
            pages = data["query"]["pages"]

            page = next(iter(pages.values()))

            return {
                "Judul": page.get("title", judul),
                "API_Text": page.get("extract", "")
            }

        # Terkena rate limit
        elif response.status_code == 429:

            retry_after = response.headers.get("Retry-After")

            if retry_after:
                wait_time = int(retry_after)
            else:
                wait_time = 5 * (attempt + 1)

            print(
                f"429 untuk '{judul}'. "
                f"Menunggu {wait_time} detik..."
            )

            time.sleep(wait_time)

        # Error lainnya
        else:

            print(
                f"Error {response.status_code} "
                f"untuk '{judul}'"
            )

            time.sleep(5)

    return {
        "Judul": judul,
        "API_Text": ""
    }

Uji coba API pada 5 artikel

In [ ]:
test_single = get_wikipedia_api_safe("Algoritma Lanczos")

print("Judul:", test_single["Judul"])
print("Panjang:", len(test_single["API_Text"]))
print()
print(test_single["API_Text"][:500])

Judul: Algoritma Lanczos
Panjang: 24279

Algoritma Lanczos adalah algoritma iteratif adaptasi dari metode daya (power method) untuk menemukan 
  
    
      
        m
      
    
    {\displaystyle m}
  
 nilai dan vektor eigen yang "paling berguna" (umumnya yang tertinggi/terendah) dari sebuah matriks Hermite berukuran 
  
    
      
        n
        ×
        n
      
    
    {\displaystyle n\times n}
  
, dengan 
  
    
      
        m
      
    
    {\displaystyle m}
  
 tidak perlu jauh lebih kecil dari 
  
    
      
    


In [ ]:
test_titles = df_final["Judul"].head(5).tolist()

test_results = []

for judul in test_titles:

    print(f"Mengambil: {judul}")

    hasil = get_wikipedia_api_safe(judul)
    test_results.append(hasil)

    print(f"Panjang teks: {len(hasil['API_Text'])}")

    # jeda antar request
    time.sleep(2)

Mengambil: Algoritma bias
Panjang teks: 9922
Mengambil: Algoritma Lanczos
Panjang teks: 24279
Mengambil: ASCII
Panjang teks: 2381
Mengambil: Karakter null
Panjang teks: 1768
Mengambil: Komputasi optis
Panjang teks: 1598


In [ ]:
df_api_test = pd.DataFrame(test_results)

print(df_api_test[["Judul", "API_Text"]])

print(
    "Jumlah artikel:",
    len(df_api_test)
)

print(
    "Teks kosong:",
    (df_api_test["API_Text"].str.len() == 0).sum()
)

               Judul                                           API_Text
0     Algoritma bias  Algoritma bias menggambarkan galat sistematis ...
1  Algoritma Lanczos  Algoritma Lanczos adalah algoritma iteratif ad...
2              ASCII  (ASCII) singkatan dari American Standard Code ...
3      Karakter null  Karakter null (atau bita null), disingkat NUL ...
4    Komputasi optis  Komputasi optis adalah penggunaan foton yang d...
Jumlah artikel: 5
Teks kosong: 0


Ambil 100 Artikel dari df_final

In [ ]:
api_articles = []

for i, judul in enumerate(df_final["Judul"], start=1):

    print(f"[{i}/100] Mengambil: {judul}")

    hasil = get_wikipedia_api_safe(judul)
    api_articles.append(hasil)

    print(f"    Panjang teks: {len(hasil['API_Text'])}")

    # jeda agar tidak terlalu cepat
    time.sleep(2)

[1/100] Mengambil: Algoritma bias
    Panjang teks: 9922
[2/100] Mengambil: Algoritma Lanczos
    Panjang teks: 24279
[3/100] Mengambil: ASCII
    Panjang teks: 2381
[4/100] Mengambil: Karakter null
    Panjang teks: 1768
[5/100] Mengambil: Komputasi optis
    Panjang teks: 1598
[6/100] Mengambil: Membangkitkan Kombinasi
429 untuk 'Membangkitkan Kombinasi'. Menunggu 29 detik...
    Panjang teks: 3260
[7/100] Mengambil: Komputasi neuromorfik
    Panjang teks: 2464
[8/100] Mengambil: Teori otomata
    Panjang teks: 6619
[9/100] Mengambil: Kecerdasan buatan
    Panjang teks: 11028
[10/100] Mengambil: Agen otonom
    Panjang teks: 2375
[11/100] Mengambil: Algoritma genetik
    Panjang teks: 13458
[12/100] Mengambil: Algoritma semut
    Panjang teks: 2308
[13/100] Mengambil: Apple Intelligence
    Panjang teks: 4022
[14/100] Mengambil: Claude (model bahasa)
    Panjang teks: 5646
[15/100] Mengambil: Dampak lingkungan dari kecerdasan buatan
    Panjang teks: 8203
[16/100] Mengambil: Etika da

Buat DataFrame API

In [ ]:
df_api = pd.DataFrame(api_articles)

print(df_api.shape)
print(df_api.head())

(100, 2)
               Judul                                           API_Text
0     Algoritma bias  Algoritma bias menggambarkan galat sistematis ...
1  Algoritma Lanczos  Algoritma Lanczos adalah algoritma iteratif ad...
2              ASCII  (ASCII) singkatan dari American Standard Code ...
3      Karakter null  Karakter null (atau bita null), disingkat NUL ...
4    Komputasi optis  Komputasi optis adalah penggunaan foton yang d...


Validasi hasil API

In [ ]:
print("Jumlah artikel:", len(df_api))
print("Judul unik:", df_api["Judul"].nunique())
print("Teks kosong:", (df_api["API_Text"].str.len() == 0).sum())
print("Teks null:", df_api["API_Text"].isna().sum())

Jumlah artikel: 100
Judul unik: 100
Teks kosong: 0
Teks null: 0


In [ ]:
df_api["Panjang_API"] = df_api["API_Text"].str.len()

print(df_api["Panjang_API"].describe())

count      100.000000
mean      8482.050000
std       6945.305493
min       1598.000000
25%       3467.750000
50%       5845.500000
75%      12115.250000
max      33395.000000
Name: Panjang_API, dtype: float64


### Extracting hasil scraping API to .csv

In [112]:
df_api_export = df_api[['Judul', 'API_Text']].copy()

df_api_export['topik'] = df_api_export['Judul'].map(kategori_final)

df_api_export.columns = ['judul', 'teks', 'topik']

df_api_export.insert(0, 'no', range(1, len(df_api_export) + 1))

df_api_export.to_csv('hasil_scraping_api.csv', index=False)

print("Berhasil menyimpan df_api dengan kolom topik ke 'hasil_scraping_api.csv'!")
print(df_api_export.head())

Berhasil menyimpan df_api dengan kolom topik ke 'hasil_scraping_api.csv'!
   no              judul                                               teks  \
0   1     Algoritma bias  Algoritma bias menggambarkan galat sistematis ...   
1   2  Algoritma Lanczos  Algoritma Lanczos adalah algoritma iteratif ad...   
2   3              ASCII  (ASCII) singkatan dari American Standard Code ...   
3   4      Karakter null  Karakter null (atau bita null), disingkat NUL ...   
4   5    Komputasi optis  Komputasi optis adalah penggunaan foton yang d...   

              topik  
0  Computer Science  
1  Computer Science  
2  Computer Science  
3  Computer Science  
4  Computer Science  


## Gabungan HTTP + API

In [ ]:
df_compare = df_final.merge(
    df_api,
    on="Judul",
    how="inner",
    validate="one_to_one"
)

In [ ]:
print(df_compare.shape)
print(df_compare.columns)

(100, 10)
Index(['Judul', 'URL', 'Overview_Isi', 'Tanggal_Suntingan', 'Penerbit',
       'Panjang_Teks', 'Topik_Sementara', 'Topik', 'API_Text', 'Panjang_API'],
      dtype='object')


In [ ]:
print("Jumlah data HTTP :", len(df_final))
print("Jumlah data API  :", len(df_api))
print("Jumlah hasil merge:", len(df_compare))
print("Judul unik hasil merge:", df_compare["Judul"].nunique())

Jumlah data HTTP : 100
Jumlah data API  : 100
Jumlah hasil merge: 100
Judul unik hasil merge: 100


In [ ]:
df_compare[
    [
        "Judul",
        "Topik",
        "Panjang_Teks",
        "Panjang_API"
    ]
].head(10)

,Judul,Topik,Panjang_Teks,Panjang_API
0,Algoritma bias,Computer Science,9968,9922
1,Algoritma Lanczos,Computer Science,10448,24279
2,ASCII,Computer Science,2153,2381
3,Karakter null,Computer Science,1762,1768
4,Komputasi optis,Computer Science,1552,1598
5,Membangkitkan Kombinasi,Computer Science,1516,3260
6,Komputasi neuromorfik,Computer Science,1598,2464
7,Teori otomata,Computer Science,1972,6619
8,Kecerdasan buatan,AI,7925,11028
9,Agen otonom,AI,2025,2375


In [ ]:
df_compare["Selisih_Panjang"] = (
    df_compare["Panjang_API"]
    - df_compare["Panjang_Teks"]
)

In [ ]:
df_compare[
    [
        "Judul",
        "Topik",
        "Panjang_Teks",
        "Panjang_API",
        "Selisih_Panjang"
    ]
].head(10)

,Judul,Topik,Panjang_Teks,Panjang_API,Selisih_Panjang
0,Algoritma bias,Computer Science,9968,9922,-46
1,Algoritma Lanczos,Computer Science,10448,24279,13831
2,ASCII,Computer Science,2153,2381,228
3,Karakter null,Computer Science,1762,1768,6
4,Komputasi optis,Computer Science,1552,1598,46
5,Membangkitkan Kombinasi,Computer Science,1516,3260,1744
6,Komputasi neuromorfik,Computer Science,1598,2464,866
7,Teori otomata,Computer Science,1972,6619,4647
8,Kecerdasan buatan,AI,7925,11028,3103
9,Agen otonom,AI,2025,2375,350


*Perbandingan Panjang Teks HTTP dan API*

Perbandingan dilakukan dengan menghitung selisih panjang teks antara hasil ekstraksi HTTP dan hasil ekstraksi API. Nilai positif menunjukkan bahwa teks hasil API lebih panjang, sedangkan nilai negatif menunjukkan bahwa teks hasil HTTP lebih panjang.

In [ ]:
print("Rata-rata panjang HTTP:",
      df_compare["Panjang_Teks"].mean())

print("Rata-rata panjang API:",
      df_compare["Panjang_API"].mean())

print("Rata-rata selisih:",
      df_compare["Selisih_Panjang"].mean())

Rata-rata panjang HTTP: 6638.64
Rata-rata panjang API: 8482.05
Rata-rata selisih: 1843.41


### Extracting hasil compare HTTP API to .csv

In [ ]:
df_compare_export = df_compare[['Judul', 'API_Text', 'Overview_Isi', 'Topik']].copy()

df_compare_export.columns = ['judul', 'teks_api', 'teks_html', 'topik']

df_compare_export.insert(0, 'no', range(1, len(df_compare_export) + 1))

df_compare_export.to_csv('hasil_scraping_compare.csv', index=False)

print("Berhasil menyimpan data df_compare ke 'hasil_scraping_compare.csv'!")
print(df_compare_export.head())

Berhasil menyimpan data df_compare ke 'hasil_scraping_compare.csv'!
   no              judul                                           teks_api  \
0   1     Algoritma bias  Algoritma bias menggambarkan galat sistematis ...   
1   2  Algoritma Lanczos  Algoritma Lanczos adalah algoritma iteratif ad...   
2   3              ASCII  (ASCII) singkatan dari American Standard Code ...   
3   4      Karakter null  Karakter null (atau bita null), disingkat NUL ...   
4   5    Komputasi optis  Komputasi optis adalah penggunaan foton yang d...   

                                           teks_html             topik  
0  Algoritma bias menggambarkan galat sistematis ...  Computer Science  
1  Algoritma Lanczos adalah algoritma iteratif ad...  Computer Science  
2  (ASCII) singkatan dari American Standard Code ...  Computer Science  
3  Karakter null (atau bita null), disingkat NUL ...  Computer Science  
4  Komputasi optis adalah penggunaan foton yang d...  Computer Science  
